# Week 16 — Model-Based RL (Dreamer-lite)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, torch
from project import solutions as S

## Ex 1 (★) KL between diagonal Gaussians
> **Hint**: KL = 0.5 * (var_q/var_p + (mu_q-mu_p)^2/var_p + 2 log(std_p/std_q) - 1) summed.

In [ ]:
def kl(mu_q, std_q, mu_p, std_p):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
kl = S.exercise_1_kl_diag_gaussian
mu, std = torch.zeros(3), torch.ones(3)
assert float(kl(mu, std, mu, std).sum()) < 1e-4

## Ex 2 (★) Lambda-return
> **Hint**: backward recursion G_t = r_t + gamma c_t [(1-lam) V_{t+1} + lam G_{t+1}].

In [ ]:
def lam_ret(r, v, c, gamma, lam):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
lam_ret = S.exercise_2_lambda_return
H, B = 3, 1
rs = [torch.ones(B) for _ in range(H)]
vs = [torch.zeros(B) for _ in range(H+1)]
cs = [torch.ones(B) for _ in range(H)]
G = lam_ret(rs, vs, cs, gamma=0.99, lam=1.0)
assert abs(float(G[0,0]) - (1 + 0.99 + 0.99**2)) < 1e-3

## Ex 3 (★★) RSSM imagine one step
> **Hint**: use the reference helper.

In [ ]:
def rssm_step(seed=0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
rssm_step = S.exercise_3_rssm_imagine
out = rssm_step(0)
assert np.isfinite(out['feat_norm'])

## Ex 4 (★★) Train Dreamer-lite
> **Hint**: 12 iterations should yield improvement.

In [ ]:
def train_dreamer(seed=0, n_iters=12):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
train_dreamer = S.exercise_4_train_dreamer
out = train_dreamer(0, 5)
assert out['eval_curve'].shape == (5, 2)

## Ex 5 (★★★) KL-balancing trick
> **Hint**: alpha * KL(stop(post) || prior) + (1-alpha) * KL(post || stop(prior)).

In [ ]:
def kl_bal(mu_q, std_q, mu_p, std_p, alpha=0.8):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
kl_bal = S.exercise_5_kl_balanced
mu = torch.zeros(2, requires_grad=True); std = torch.ones(2, requires_grad=True)
mu2 = torch.ones(2, requires_grad=True); std2 = torch.ones(2, requires_grad=True)
v = kl_bal(mu, std, mu2, std2, 0.8).sum(); v.backward()
assert mu.grad is not None

## Ex 6 (★★) Improvement summary
> **Hint**: contrast first vs last eval return.

In [ ]:
def improvement(out):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
improvement = S.exercise_6_imagined_vs_real
print(improvement(out))

## Ex 7 (★★) Loss components
> **Hint**: just return final eval from a short Dreamer run.

In [ ]:
def loss_comp(seed=0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
loss_comp = S.exercise_7_world_model_loss_components
print(loss_comp(0))

## Ex 8 (★★★, meaningful objective) Recruitment world-model essay
Write a short paragraph on why a Dreamer-style world model is the right abstraction for recruitment dynamics; compare to model-free Q-learning.

In [ ]:
def essay():
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
essay = S.exercise_8_recruitment_world_model_demo
print(essay()[:200], '...')

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

_A consolidated cheat-sheet — read only after you've attempted each exercise. Stronger than the inline hints above the scaffolds, weaker than the full solution. Worked answers are in `project/solutions.py` in the same folder (`week-16-model-based`).
_

### Ex 1 (★) KL between diagonal Gaussians

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Ex 2 (★) Lambda-return

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Ex 3 (★★) RSSM imagine one step

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Ex 4 (★★) Train Dreamer-lite

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Ex 5 (★★★) KL-balancing trick

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Ex 6 (★★) Improvement summary

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Ex 7 (★★) Loss components

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Ex 8 (★★★, meaningful objective) Recruitment world-model essay

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.


---
_End of week. Mark this notebook in your tracker only after every exercise's hidden test cell passes._
